# Model 3: LightGBM Classifier with TF-IDF Text Similarities

## Overview
This notebook implements a **classical machine learning pipeline** using LightGBM as the classifier. It serves as the "Additional model of choice" for the MCQ Solver project, providing an interpretable, fast-training counterpart to the deep-learning models.

## Pipeline Summary
1. **Feature Engineering** — Extract TF-IDF vectors from combined context/prompt and each option (A-E), compute cosine similarity scores, and add structural numerical features (text lengths, word counts).
2. **Long-format Training** — Each row = one (question, option) pair; label = 1 if that option is the correct answer, 0 otherwise.
3. **LightGBM Binary Classifier** — Gradient-boosted tree model on engineered features with early stopping.
4. **MAP@3 Ranking** — At inference, score all 5 options per question and rank descending; top-3 form the submission string.

## Why LightGBM?
- Trains in seconds on CPU (no GPU required)
- Built-in early stopping avoids overfitting
- Feature importances are directly interpretable for viva defence
- Strong classical baseline for text-similarity ranking tasks

In [ ]:
# Cell 2: Imports
import os
import re
import numpy as np
import pandas as pd
import lightgbm as lgb
import wandb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, log_loss, f1_score
from sklearn.model_selection import train_test_split

# Authenticate W&B using the secret key stored in the Kaggle environment
WANDB_KEY = os.environ.get('WANDB_API_KEY', '')
if WANDB_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    print('[INFO] W&B API key loaded from environment.')
else:
    print('[WARN] WANDB_API_KEY not set -- W&B logging may prompt for login.')

SEED = 42
np.random.seed(SEED)
print(f'[INFO] LightGBM version : {lgb.__version__}')

In [ ]:
# Cell 3: Absolute Kaggle Data Paths
# Primary paths matching the exact Kaggle competition mount
TRAIN_PATH      = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_PATH       = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'
SAMPLE_SUB_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv'

# Local fallback resolver for offline development
def _resolve(kaggle_abs, *local_candidates):
    """Return the first path that exists; fall back to kaggle_abs on cloud."""
    if os.path.exists(kaggle_abs):
        return kaggle_abs
    for p in local_candidates:
        if os.path.exists(p):
            return p
    return kaggle_abs

TRAIN_PATH      = _resolve(TRAIN_PATH,      'data/train.csv',              '../../data/train.csv')
TEST_PATH       = _resolve(TEST_PATH,       'data/test.csv',               '../../data/test.csv')
SAMPLE_SUB_PATH = _resolve(SAMPLE_SUB_PATH, 'data/sample_submission.csv',  '../../data/sample_submission.csv')

print(f'Train      -> {TRAIN_PATH}')
print(f'Test       -> {TEST_PATH}')
print(f'Sample sub -> {SAMPLE_SUB_PATH}')

In [ ]:
# Cell 4: W&B Initialization
# All hyperparameters are logged at init for full reproducibility
CONFIG = dict(
    model_name            = 'LightGBM',
    n_estimators          = 100,
    learning_rate         = 0.05,
    num_leaves            = 31,
    max_depth             = -1,
    min_child_samples     = 20,
    tfidf_max_features    = 5000,
    early_stopping_rounds = 15,
    test_size             = 0.2,
    seed                  = SEED,
)

run = wandb.init(
    project = '23f2004343-t22026',
    name    = 'Model_3_LightGBM',
    config  = CONFIG,
)
print('[INFO] W&B run initialized:', run.name)

In [ ]:
# Cell 5: Feature Engineering Pipeline
# Viva note:
#   We combine context + prompt into a query string, vectorize everything with
#   one shared TF-IDF vocabulary, then compute cosine similarity between each
#   query vector and each option vector. This scalar 'relevance' score is the
#   primary feature. Supplementary structural features (lengths, word counts)
#   capture surface-level patterns that cosine similarity misses.

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']

def clean_text(text):
    """Lowercase and normalise whitespace for TF-IDF preprocessing."""
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


# Load raw CSVs
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

for df in [train_raw, test_raw]:
    if 'context' not in df.columns:
        df['context'] = ''
    df['context'] = df['context'].fillna('')
    for col in OPTION_COLS:
        df[col] = df[col].fillna('')

print(f'[Data] Train shape: {train_raw.shape}')
print(f'[Data] Test  shape: {test_raw.shape}')
print(f'[Data] Train columns: {list(train_raw.columns)}')

# Build query strings: context + prompt
train_queries = (train_raw['context'] + ' ' + train_raw['prompt']).apply(clean_text).tolist()
test_queries  = (test_raw['context']  + ' ' + test_raw['prompt']).apply(clean_text).tolist()

# Collect all text to fit one shared TF-IDF vocabulary (no test leakage)
all_option_texts = []
for df in [train_raw, test_raw]:
    for col in OPTION_COLS:
        all_option_texts.extend(df[col].apply(clean_text).tolist())

corpus = train_queries + test_queries + all_option_texts

# Shared vectorizer: unigrams + bigrams, log-scaled TF, top-5000 features
vectorizer = TfidfVectorizer(
    max_features = CONFIG['tfidf_max_features'],
    stop_words   = 'english',
    ngram_range  = (1, 2),
    sublinear_tf = True,
)
vectorizer.fit(corpus)
print(f'[TF-IDF] Vocabulary size: {len(vectorizer.vocabulary_):,}')


def build_features(df, queries):
    """Return long-format DataFrame: one row per (question, option) pair.

    Features:
        cosine_similarity  -- TF-IDF cosine sim between query and option
        prompt_len         -- character count of the prompt
        option_len         -- character count of the option text
        len_ratio          -- option_len / (prompt_len + 1)
        word_count_prompt  -- word count of the prompt
        word_count_option  -- word count of the option
        word_count_diff    -- absolute difference in word counts
        option_idx         -- 0-based option index (A=0 ... E=4)
    """
    query_vecs = vectorizer.transform([clean_text(q) for q in queries])

    rows = []
    for row_idx, (_, row) in enumerate(df.iterrows()):
        q_vec      = query_vecs[row_idx]
        prompt_txt = str(row['prompt'])
        prompt_len = len(prompt_txt)
        wc_prompt  = len(prompt_txt.split())

        for opt_idx, opt_key in enumerate(OPTION_COLS):
            opt_text = clean_text(row[opt_key])
            opt_vec  = vectorizer.transform([opt_text])
            cos_sim  = float(cosine_similarity(q_vec, opt_vec)[0][0])

            opt_len = len(opt_text)
            wc_opt  = len(opt_text.split())

            label = 0
            if 'answer' in row:
                label = 1 if str(row['answer']).strip().upper() == opt_key else 0

            rows.append({
                'id':               row['id'],
                'opt_key':          opt_key,
                'cosine_similarity':cos_sim,
                'prompt_len':       prompt_len,
                'option_len':       opt_len,
                'len_ratio':        opt_len / (prompt_len + 1),
                'word_count_prompt':wc_prompt,
                'word_count_option':wc_opt,
                'word_count_diff':  abs(wc_prompt - wc_opt),
                'option_idx':       opt_idx,
                'label':            label,
            })

    return pd.DataFrame(rows)


print('[Features] Extracting training features...')
train_feat = build_features(train_raw, train_queries)
print(f'[Features] Train long-format shape: {train_feat.shape}')

print('[Features] Extracting test features...')
test_feat  = build_features(test_raw, test_queries)
print(f'[Features] Test  long-format shape: {test_feat.shape}')

In [ ]:
# Cell 6: Prepare Long-Format Training and Validation Splits
# Viva note:
#   Long format: each question (5 options) contributes 5 rows.
#   Binary label = 1 for the correct option, 0 for all four distractors.
#   We split at the QUESTION level so all 5 option rows for a given question
#   land in the same split, preventing any data leakage.

FEATURE_COLS = [
    'cosine_similarity',
    'prompt_len',
    'option_len',
    'len_ratio',
    'word_count_prompt',
    'word_count_option',
    'word_count_diff',
    'option_idx',
]

# Split at question level
unique_ids = train_raw['id'].unique()
train_ids, val_ids = train_test_split(
    unique_ids,
    test_size    = CONFIG['test_size'],
    random_state = SEED,
)

train_split = train_feat[train_feat['id'].isin(train_ids)].reset_index(drop=True)
val_split   = train_feat[train_feat['id'].isin(val_ids)].reset_index(drop=True)

X_train, y_train = train_split[FEATURE_COLS], train_split['label']
X_val,   y_val   = val_split[FEATURE_COLS],   val_split['label']

print(f'[Split] Train rows: {len(X_train):,}  ({len(train_ids):,} questions)')
print(f'[Split] Val   rows: {len(X_val):,}  ({len(val_ids):,} questions)')
print(f'[Split] Positive label rate -- Train: {y_train.mean():.4f} | Val: {y_val.mean():.4f}')

In [ ]:
# Cell 7: Train LightGBM Classifier and Log Metrics to W&B
# Viva note:
#   LightGBM uses leaf-wise tree growth (faster than level-wise in XGBoost).
#   binary_logloss = cross-entropy on binary labels, equivalent to BCELoss.
#   Early stopping monitors validation logloss and halts when it plateaus,
#   preventing overfitting without manual epoch tuning.

model = lgb.LGBMClassifier(
    objective         = 'binary',
    metric            = 'binary_logloss',
    n_estimators      = CONFIG['n_estimators'],
    learning_rate     = CONFIG['learning_rate'],
    num_leaves        = CONFIG['num_leaves'],
    max_depth         = CONFIG['max_depth'],
    min_child_samples = CONFIG['min_child_samples'],
    random_state      = SEED,
    verbose           = -1,
)

model.fit(
    X_train, y_train,
    eval_set  = [(X_train, y_train), (X_val, y_val)],
    callbacks = [
        lgb.early_stopping(stopping_rounds=CONFIG['early_stopping_rounds'], verbose=False),
        lgb.log_evaluation(period=10),
    ],
)

best_iter = model.best_iteration_
print(f'[Train] Best iteration: {best_iter}')

# Compute predictions for both splits
train_proba = model.predict_proba(X_train)[:, 1]
val_proba   = model.predict_proba(X_val)[:, 1]
train_preds = (train_proba > 0.5).astype(int)
val_preds   = (val_proba   > 0.5).astype(int)

train_acc = accuracy_score(y_train, train_preds)
val_acc   = accuracy_score(y_val,   val_preds)
train_ll  = log_loss(y_train, train_proba)
val_ll    = log_loss(y_val,   val_proba)
train_f1  = f1_score(y_train, train_preds, average='macro', zero_division=0)
val_f1    = f1_score(y_val,   val_preds,   average='macro', zero_division=0)

print(f'Train -- Acc: {train_acc:.4f} | LogLoss: {train_ll:.4f} | F1: {train_f1:.4f}')
print(f'Val   -- Acc: {val_acc:.4f} | LogLoss: {val_ll:.4f} | F1: {val_f1:.4f}')

# MAP@3 on validation set
def map_at_3(pred_lists, true_labels):
    """Mean Average Precision @ 3."""
    scores = []
    for preds, truth in zip(pred_lists, true_labels):
        scores.append(1.0 / (preds.index(truth) + 1) if truth in preds else 0.0)
    return float(np.mean(scores)) if scores else 0.0

val_copy = val_split.copy()
val_copy['proba'] = val_proba
gt_map = train_raw.set_index('id')['answer'].to_dict() if 'answer' in train_raw.columns else {}

val_pred_lists, val_true_labels = [], []
for q_id, grp in val_copy.groupby('id', sort=False):
    ranked = grp.sort_values('proba', ascending=False)
    top3   = ranked['opt_key'].head(3).tolist()
    gt     = str(gt_map.get(q_id, '')).strip().upper()
    val_pred_lists.append(top3)
    val_true_labels.append(gt)

val_map3 = map_at_3(val_pred_lists, val_true_labels)
print(f'Val   -- MAP@3: {val_map3:.4f}')

# Log all metrics and feature importance to W&B
wandb.log({
    'best_iteration': best_iter,
    'train_accuracy': train_acc,
    'val_accuracy':   val_acc,
    'train_logloss':  train_ll,
    'val_logloss':    val_ll,
    'train_f1':       train_f1,
    'val_f1':         val_f1,
    'val_map_at_3':   val_map3,
})
wandb.summary['best_val_map_at_3'] = val_map3

fi_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)
print('\n[Feature Importances]')
print(fi_df.to_string(index=False))
wandb.log({'feature_importance': wandb.Table(dataframe=fi_df)})

In [ ]:
# Cell 8: Inference on Test Set and Submission Generation
# Viva note:
#   For each test question, the model outputs a probability for each of the
#   5 options. We sort them descending and take the top-3 option keys,
#   joining with a space to form the submission string (e.g. 'C A D').
#   This matches the MAP@3 evaluation format required by the competition.

X_test = test_feat[FEATURE_COLS]
test_feat['proba'] = model.predict_proba(X_test)[:, 1]

# Group by question ID, rank options by score descending, take top-3
predictions = []
for q_id, grp in test_feat.groupby('id', sort=False):
    ranked    = grp.sort_values('proba', ascending=False)
    top3_opts = ranked['opt_key'].head(3).tolist()
    predictions.append({'id': q_id, 'prediction': ' '.join(top3_opts)})

submission_df = pd.DataFrame(predictions)

# Verify IDs match sample submission
if os.path.exists(SAMPLE_SUB_PATH):
    sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
    assert set(submission_df['id'].tolist()) == set(sample_sub['id'].tolist()), \
        'ID mismatch between submission and sample_submission.csv!'
    print('[Verify] All test IDs matched with sample_submission.csv OK')

OUTPUT_PATH = 'submission.csv'
submission_df.to_csv(OUTPUT_PATH, index=False)
print(f'[Output] Saved to {OUTPUT_PATH} -- {len(submission_df):,} rows')
print(submission_df.head(10).to_string(index=False))

# Log submission file as W&B artifact for traceability
artifact = wandb.Artifact('submission_lgbm', type='dataset')
artifact.add_file(OUTPUT_PATH)
wandb.log_artifact(artifact)

wandb.finish()
print('[Done] W&B run closed.')